In [1]:
import geopandas as gpd
import osmnx as ox
import pathlib 
NOTEBOOK_PATH = pathlib.Path().resolve()
DATA_DIRECTORY = NOTEBOOK_PATH / "data"
OUTPUT_DIRECTORY = NOTEBOOK_PATH / "docs"
NOTEBOOK_PATH

PosixPath('/home/jovyan/my-work/practical4')

In [2]:
# Import shapefile of study area as a geodataframe
studyarea = gpd.read_file(DATA_DIRECTORY/"aluerajaus"/"aluerajaus.shp")

# Setting to CRS on WGS84 (EPSG:4326), as that is the requirement for OSMnx 
if studyarea.crs != "EPSG:4326":
    studyarea = studyarea.to_crs("EPSG:4326")

# Selecting the geometry of the study area
studyarea_polygon = studyarea.geometry.iloc[0]

# Additional attributes to be requested to be downloaded for the cycling network
extra_tags = [
    "bicycle",
    "cycleway",
    "cycleway:left",
    "cycleway:right",
    "cycleway:both",
    "foot",
    "segregated",
    "surface",
    "tunnel",
    "bridge",
    "layer",
    "covered"
]
ox.settings.useful_tags_way = list(
    set(ox.settings.useful_tags_way + extra_tags)
)

In [3]:
# Import cycleable streets inside the study area from OSM
G = ox.graph_from_polygon(studyarea_polygon, network_type="bike", simplify=True, retain_all=True)

# nodes, edges = ox.graph_to_gdfs(G)

In [4]:
# Exclude unnecessary parking roads etc
G_clean = G.copy()

excluded_services = {
    "parking_aisle",
    "parking",
    "driveway",
    "emergency_access"
}

def has_excluded_tag(value, excluded):
    if isinstance(value, (list, tuple, set)):
        return any(v in excluded for v in value)
    return value in excluded

edges_to_remove = [
    (u, v, key)
    for u, v, key, data in G_clean.edges(
        keys=True, data=True
    )
    if has_excluded_tag(
        data.get("service"),
        excluded_services
    )
]

G_clean.remove_edges_from(edges_to_remove)

print(f"Removed {len(edges_to_remove)} directed edges")

Removed 23467 directed edges


In [5]:
# Remove isolated nodes
G_clean.remove_nodes_from(list(nx.isolates(G_clean)))

NameError: name 'nx' is not defined

In [ ]:
nodes_c, edges_c = ox.graph_to_gdfs(G_clean)
#edges_c.explore()

In [ ]:
# Save the bike network graph
ox.io.save_graphml(
    G_clean,
    filepath=OUTPUT_DIRECTORY/"turku_bike_network.graphml"
)

In [ ]:
"""# Save nodes and edges for spatial analysis
ox.io.save_graph_geopackage(
    G_clean,
    filepath=docs/"turku_bike_network.gpkg",
    directed=True
)"""

In [ ]:
"""edges_c.to_file(
    "cycling_network.gpkg",
    layer="cycling_network",
    driver="GPKG"
)"""

In [ ]:
"""edges_3067=edges_c.to_crs(3067)
edges_3067.to_file(
    "cycling_network_3067.gpkg",
    layer="cycling_network",
    driver="GPKG"
)"""

In [ ]:
# The graphml can be reloaded with
# G = ox.io.load_graphml("turku_bike_network.graphml")